# ZDB

In [382]:
##########################################
##### KI-generierte Helfer-Funktion ######
##########################################

from dataclasses import dataclass

import requests
from lxml import etree


NS = {
    "sru": "http://www.loc.gov/zing/srw/",
    "diag": "http://www.loc.gov/zing/srw/diagnostic/",
}


@dataclass
class SRUResult:
    records: list[etree._Element] | list[str]
    total_hits: int
    next_start_record: int | None  # None means no next page
    url: str


class SRUClient:
    def __init__(self, url, record_schema="MARC21plus-xml", timeout=30):
        self.url = url
        self.record_schema = record_schema
        self.timeout = timeout

    def search(
        self,
        query: str,
        *,
        start_record: int = 1,
        maximum_records: int = 100,
        as_strings: bool = False,
        sru_version: str = "1.1"
    ) -> SRUResult:
        if start_record < 1 or maximum_records < 1:
            raise ValueError("start_record and maximum_records must be >= 1")

        response = requests.get(
            self.url,
            params={
                "version": sru_version,
                "operation": "searchRetrieve",
                "recordSchema": self.record_schema,
                "recordPacking": "xml",
                "maximumRecords": maximum_records,
                "query": query,
                "startRecord": start_record,
            },
            timeout=self.timeout,
        )
        response.raise_for_status()
        print(response.url)

        # Parse bytes so the XML declaration determines the encoding.
        parser = etree.XMLParser(resolve_entities=False, no_network=True)
        root = etree.fromstring(response.content, parser=parser)

        # SRU errors can be returned with HTTP status 200.
        diagnostics = root.findall(".//diag:diagnostic", namespaces=NS)
        if diagnostics:
            messages = [
                " ".join(t.strip() for t in d.itertext() if t.strip())
                for d in diagnostics
            ]
            raise RuntimeError("SRU diagnostic: " + "; ".join(messages))

        total_text = root.findtext("sru:numberOfRecords", namespaces=NS)
        if total_text is None:
            raise ValueError("Response is missing sru:numberOfRecords")
        total_hits = int(total_text)

        envelopes = root.findall("sru:records/sru:record", namespaces=NS)
        elements = []

        for envelope in envelopes:
            # Return the payload, not the surrounding SRU record wrapper.
            payloads = envelope.xpath("./sru:recordData/*", namespaces=NS)
            if len(payloads) != 1:
                raise ValueError("Expected one XML payload per SRU record")
            elements.append(payloads[0])

        next_text = root.findtext("sru:nextRecordPosition", namespaces=NS)
        if next_text is not None:
            next_start = int(next_text)
        else:
            # Some servers omit nextRecordPosition. Advance by the number
            # actually returned, which may be less than maximum_records.
            candidate = start_record + len(envelopes)
            next_start = (
                candidate if envelopes and candidate <= total_hits else None
            )

        records = (
            [
                etree.tostring(element, encoding="unicode", with_tail=False)
                for element in elements
            ]
            if as_strings
            else elements
        )

        return SRUResult(
            records=records,
            total_hits=total_hits,
            next_start_record=next_start,
            url=response.url
        )

## Welche besonderen Zeitschriften hat die UB der FU?

In [385]:
client = SRUClient(url)

result = client.search(
    query,
    start_record=1,
    maximum_records=100,
    as_strings=False
)

# Fetch the following page:
result_collector = result.records

while True:
    if result.next_start_record is not None:
        result = client.search(
            query,
            start_record=result.next_start_record,
            as_strings=False,
        )
        result_collector.extend(result.records)
    else:
        break

https://services.dnb.de/sru/zdb?version=1.1&operation=searchRetrieve&recordSchema=MARC21plus-xml&recordPacking=xml&maximumRecords=100&query=dnb.isil%3DDE-188+AND+dnb.dok%3DZeitschrift+AND+dnb.dcz%3D890&startRecord=1
https://services.dnb.de/sru/zdb?version=1.1&operation=searchRetrieve&recordSchema=MARC21plus-xml&recordPacking=xml&maximumRecords=100&query=dnb.isil%3DDE-188+AND+dnb.dok%3DZeitschrift+AND+dnb.dcz%3D890&startRecord=101
https://services.dnb.de/sru/zdb?version=1.1&operation=searchRetrieve&recordSchema=MARC21plus-xml&recordPacking=xml&maximumRecords=100&query=dnb.isil%3DDE-188+AND+dnb.dok%3DZeitschrift+AND+dnb.dcz%3D890&startRecord=101
https://services.dnb.de/sru/zdb?version=1.1&operation=searchRetrieve&recordSchema=MARC21plus-xml&recordPacking=xml&maximumRecords=100&query=dnb.isil%3DDE-188+AND+dnb.dok%3DZeitschrift+AND+dnb.dcz%3D890&startRecord=201
https://services.dnb.de/sru/zdb?version=1.1&operation=searchRetrieve&recordSchema=MARC21plus-xml&recordPacking=xml&maximumRecords=

In [386]:
NSMAP = { 'marc' : 'http://www.loc.gov/MARC21/slim'}

for r in result_collector:
    title = r.xpath('marc:record[@type="Bibliographic"]/marc:datafield[@tag="245"]/marc:subfield[@code="a"]/text()', namespaces = NSMAP)[0]
    holding_count = len(r.xpath('marc:record[@type="Holdings"]', namespaces = NSMAP ))
    if holding_count < 3:
        print(f"- {title} ({holding_count})")

- Littéraire (2)
- Dydaktyka literatury polskiej i je̜zyka polskiego (2)
- Bulletin des publications en langues slaves, intéressant les études byzantines (1)
- Papyri (1)
- Japan English books in print (2)
- Geschäftsbericht ... der Dr. Lepsius Deutschen Orient-Mission (2)
